# EC. Primer Parcial — Proyecto Práctico HPC
## Implementación secuencial

**Asignatura:** Cómputo de Alto Rendimiento  
**Integrante 2:** Diego Francisco Almaraz Vargas  

En este notebook se implementa una versión paralela del programa
secuencial utilizando múltiples trabajadores (workers).

Se evaluará el tiempo de ejecución utilizando:

- 1 worker
- 2 workers
- 4 workers

Cada configuración se ejecutará 3 veces para obtener resultados
más representativos.

## Estrategia de paralelización

La versión paralela divide el trabajo en diferentes tareas que pueden
ser ejecutadas de manera simultánea.

Para realizar la paralelización se utilizará `ProcessPoolExecutor`,
que permite ejecutar tareas utilizando diferentes procesos de Python.

Se probarán diferentes cantidades de workers para observar cómo cambia
el tiempo de ejecución al aumentar el paralelismo.

In [3]:
import time
import numpy as np

from concurrent.futures import ProcessPoolExecutor

In [4]:
# Configuración del experimento

REPETICIONES = 3
WORKERS = [1, 2, 4]

print("Repeticiones:", REPETICIONES)
print("Workers:", WORKERS)

Repeticiones: 3
Workers: [1, 2, 4]


In [11]:
from sklearn.datasets import fetch_covtype

print("Cargando Covertype...")

covtype = fetch_covtype(as_frame=True)

X = covtype.data

print("Dataset cargado correctamente.")
print("Número de filas:", X.shape[0])
print("Número de características:", X.shape[1])

Cargando Covertype...
Dataset cargado correctamente.
Número de filas: 581012
Número de características: 54


In [12]:
X_np = X.to_numpy(dtype=np.float64)

print("Tipo:", type(X_np))
print("Forma:", X_np.shape)
print("Tipo numérico:", X_np.dtype)

Tipo: <class 'numpy.ndarray'>
Forma: (581012, 54)
Tipo numérico: float64


In [13]:
def procesar_fila(fila):
    # Aplica la operación matemática a una sola fila.
    resultado = np.sqrt(fila**2 + 1.0) + np.sin(fila)
    return np.sum(resultado)

In [14]:
resultado_prueba = procesar_fila(X_np[0])

print("Resultado de la primera fila:")
print(resultado_prueba)

print("\nTipo del resultado:")
print(type(resultado_prueba))

Resultado de la primera fila:
10349.575412542694

Tipo del resultado:
<class 'numpy.float64'>


In [15]:
def procesar_bloque(bloque):
    resultados = np.empty(bloque.shape[0], dtype=np.float64)

    for i in range(bloque.shape[0]):
        resultados[i] = procesar_fila(bloque[i])

    return resultados

In [16]:
def procesar_paralelo(datos, workers):
    # Dividir los datos en bloques
    bloques = np.array_split(datos, workers)

    # Crear los procesos
    with ProcessPoolExecutor(max_workers=workers) as executor:
        resultados_bloques = list(executor.map(procesar_bloque, bloques))

    # Unir los resultados de todos los bloques
    resultados = np.concatenate(resultados_bloques)

    return resultados

## Prueba con 1 Worker

In [17]:
resultado_paralelo_1 = procesar_paralelo(X_np, 1)

print("Cantidad de resultados:", len(resultado_paralelo_1))
print("Primer resultado:", resultado_paralelo_1[0])

Cantidad de resultados: 581012
Primer resultado: 10349.575412542694


In [18]:
def medir_paralelo(datos, workers, repeticiones=3):
    tiempos = []

    for prueba in range(repeticiones):
        inicio = time.perf_counter()

        resultados = procesar_paralelo(datos, workers)

        fin = time.perf_counter()

        tiempo = fin - inicio
        tiempos.append(tiempo)

        print(f"Prueba {prueba + 1}: {tiempo:.4f} segundos")

    return tiempos

## Medir con 1 Worker

In [19]:
tiempos_1 = medir_paralelo(X_np, 1, REPETICIONES)

promedio_1 = np.mean(tiempos_1)
desviacion_1 = np.std(tiempos_1, ddof=1)

print("\nPromedio:", promedio_1)
print("Desviación estándar:", desviacion_1)

Prueba 1: 7.5891 segundos
Prueba 2: 7.9852 segundos
Prueba 3: 7.5170 segundos

Promedio: 7.6971020183336805
Desviación estándar: 0.25206728152723806


## Medir con 2 Workers

In [20]:
tiempos_2 = medir_paralelo(X_np, 2, REPETICIONES)

promedio_2 = np.mean(tiempos_2)
desviacion_2 = np.std(tiempos_2, ddof=1)

print("\nPromedio:", promedio_2)
print("Desviación estándar:", desviacion_2)

Prueba 1: 4.9517 segundos
Prueba 2: 5.0474 segundos
Prueba 3: 4.8428 segundos

Promedio: 4.947310922999653
Desviación estándar: 0.10241547786879665


## Medir con 4 Workers

In [21]:
tiempos_4 = medir_paralelo(X_np, 4, REPETICIONES)

promedio_4 = np.mean(tiempos_4)
desviacion_4 = np.std(tiempos_4, ddof=1)

print("\nPromedio:", promedio_4)
print("Desviación estándar:", desviacion_4)

Prueba 1: 3.3661 segundos
Prueba 2: 3.3625 segundos
Prueba 3: 3.5771 segundos

Promedio: 3.435260254666597
Desviación estándar: 0.12284411048357038


## Validar

In [23]:
# Calcular los resultados de referencia usando la versión secuencial

def procesar_secuencial(datos):
    resultados = np.empty(datos.shape[0], dtype=np.float64)

    for i in range(datos.shape[0]):
        resultados[i] = procesar_fila(datos[i])

    return resultados


resultados_secuenciales = procesar_secuencial(X_np)

print("Resultados secuenciales calculados:", len(resultados_secuenciales))
print("Primer resultado:", resultados_secuenciales[0])

Resultados secuenciales calculados: 581012
Primer resultado: 10349.575412542694


In [24]:
# Validación de los resultados paralelos

resultado_validacion = procesar_paralelo(X_np, 4)

print("¿Los resultados son correctos?")
print(
    np.allclose(
        resultado_validacion,
        resultados_secuenciales,
        rtol=1e-12,
        atol=1e-12
    )
)

¿Los resultados son correctos?
True


## Preparar y guardar los datos

In [25]:
import pandas as pd

datos_paralelos = []

for i, tiempo in enumerate(tiempos_1, start=1):
    datos_paralelos.append([1, i, tiempo])

for i, tiempo in enumerate(tiempos_2, start=1):
    datos_paralelos.append([2, i, tiempo])

for i, tiempo in enumerate(tiempos_4, start=1):
    datos_paralelos.append([4, i, tiempo])

df_paralelo = pd.DataFrame(
    datos_paralelos,
    columns=["workers", "prueba", "tiempo_segundos"]
)

df_paralelo

,workers,prueba,tiempo_segundos
0,1,1,7.589091
1,1,2,7.985176
2,1,3,7.517039
3,2,1,4.951731
4,2,2,5.047445
5,2,3,4.842757
6,4,1,3.366138
7,4,2,3.362549
8,4,3,3.577093


In [26]:
df_paralelo.to_csv("../resultados/paralelo.csv", index=False)

print("Archivo guardado correctamente:")
print("../resultados/paralelo.csv")

Archivo guardado correctamente:
../resultados/paralelo.csv


## Comprobar los resultados guardados

In [27]:
df_comprobacion = pd.read_csv("../resultados/paralelo.csv")

print("Filas:", len(df_comprobacion))
print("Columnas:", list(df_comprobacion.columns))

df_comprobacion

Filas: 9
Columnas: ['workers', 'prueba', 'tiempo_segundos']


,workers,prueba,tiempo_segundos
0,1,1,7.589091
1,1,2,7.985176
2,1,3,7.517039
3,2,1,4.951731
4,2,2,5.047445
5,2,3,4.842757
6,4,1,3.366138
7,4,2,3.362549
8,4,3,3.577093


## Resultados preliminares

Se realizaron 3 pruebas para cada configuración de workers.

Los tiempos obtenidos fueron:

| Workers | Prueba 1 | Prueba 2 | Prueba 3 |
|---:|---:|---:|---:|
| 1 | 7.5891 s | 7.9852 s | 7.5170 s |
| 2 | 4.9517 s | 5.0474 s | 4.8428 s |
| 4 | 3.3661 s | 3.3625 s | 3.5771 s |

Los resultados muestran una reducción del tiempo de ejecución al utilizar
más workers en este experimento.

Los resultados numéricos de la versión paralela fueron validados contra
la versión secuencial mediante `np.allclose`, obteniendo `True`.